# MirrorTopology Step 1 — Phase D-3c：実 bank による 12 位置 profile・official gate・5 seed plan の固定（family 単位；v0.1）
受入れ済みの固定入力（D-2 の第 1 波 bank＋family reference，D-3b の追加 bank（配列検証付き受入れ））を **formal intake** で読み（bytes／配列 SHA の再検証・ledger／receipt 束縛），1 family の 3 size × 12 位置 × 2 system（36 配置 × 2）の供給を組み立て，family 共有の 5 seed BootstrapPlan／FittingPlan を**実 cluster UID から 1 回だけ**構築（登録定数 B＝2000・B_KDE＝2000・seeds 5），size input 6 → 全 size 組立て → input fingerprint → **formal 12 位置 official gate（live 環境）** → gate 後の identity 再確認 → profile record と plan identity の検証付き publication。新しい共分散・MC bank の生成，閾値／label／較正，D-2W，noise は行わない。入力は読取り専用。**High-RAM CPU runtime**（保持 payload ≈13 GB＋一時領域；51 GB が十分であることは保証されない：段階別 RSS を記録する）。既定で Drive の run を `/content/stage` に 1 回複製してから読む（intake は同じ NPZ を複数回開くため；複製後も bytes／配列 SHA の再検証は同じ）。**family ごとに 1 run**（E2 → E7 → E8）。

In [ ]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the execution target (must contain d/d3c_profile.py and registered_assets/d3b/d3b_outer_receipt.json)>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
FAMILY = 'E2'
D2_RUN_ROOT = '/content/drive/MyDrive/MirrorTopology_D2/E2_20260923T092030Z/out/d2'          # the accepted D-2 run of this family: out/d2
D3B_RUN_ROOTS = {'L1.00': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.00_20260930T010656Z/out/d3b',
                 'L1.20': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.20_20260930T054800Z/out/d3b',
                 'L1.50': '/content/drive/MyDrive/MirrorTopology_D3b/E2_L1.50_20260930T080141Z/out/d3b'}   # the three accepted D-3b partitions of this family: out/d3b
STAGE_LOCAL = True                 # copy the inputs once to /content/stage before the intake (same verification; fewer Drive reads)


In [ ]:
# --- 1. fresh scratch checkout; inventory + script + ledger + receipt bound; inputs exist and match the ledgers; Drive mounted (read-only usage)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E2','E7','E8') and set(D3B_RUN_ROOTS)=={'L1.00','L1.20','L1.50'}
from google.colab import drive; drive.mount('/content/drive')
assert os.path.isfile(f'{D2_RUN_ROOT}/d2_bank_registry.json'), D2_RUN_ROOT
for s,p in D3B_RUN_ROOTS.items(): assert os.path.isfile(f'{p}/d3_bank_registry.json'), p
SCRATCH='/content/d3c_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d3c_profile.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d3c_profile.py']
LEDGER=f'{PHASEB}/registered_assets/d3b/d3b_generation_ledger.json'; RECEIPT=f'{PHASEB}/registered_assets/d3b/d3b_outer_receipt.json'; assert sha(LEDGER)==inv['registered_assets_sha256']['registered_assets/d3b/d3b_generation_ledger.json'] and sha(RECEIPT)==inv['registered_assets_sha256']['registered_assets/d3b/d3b_outer_receipt.json']
ledger=json.load(open(LEDGER)); d2l=json.load(open(f'{PHASEB}/registered_assets/d2/d2_generation_ledger.json'))
assert D2_RUN_ROOT.rstrip('/')==d2l['families'][FAMILY]['run_root'].rstrip('/')+'/out/d2', (D2_RUN_ROOT, d2l['families'][FAMILY]['run_root'])
for s,p in D3B_RUN_ROOTS.items(): assert p.rstrip('/')==ledger['partitions'][f'{FAMILY}_{s}']['run_root'].rstrip('/')+'/d3b', (s, p)
pins=json.load(open(f'{PHASEB}/d/d3_pins.json')); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1)); assert vm.total>30e9, 'High-RAM runtime required'
OUT=f'/content/d3c_profile_{FAMILY}'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
lock=dict(commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), ledger_sha256=ledger['ledger_sha256'], receipt_sha256=json.load(open(RECEIPT))['receipt_sha256'], family=FAMILY, d2_run_root=D2_RUN_ROOT, d3b_run_roots=D3B_RUN_ROOTS, stage_local=STAGE_LOCAL, engine=inv['engine_version'], ram_total_bytes=vm.total, ram_available_bytes=vm.available); json.dump(lock, open(f'{OUT}/profile_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. optional local staging (one copy of the inputs; the script re-verifies every byte / array SHA), then the profile run (long)
t0=time.time(); IN_D2=D2_RUN_ROOT; IN_D3B=dict(D3B_RUN_ROOTS)
if STAGE_LOCAL:
    ST='/content/stage'; shutil.rmtree(ST, ignore_errors=True); os.makedirs(ST)
    IN_D2=f'{ST}/d2'; shutil.copytree(D2_RUN_ROOT, IN_D2)
    IN_D3B={s: f'{ST}/d3b_{s}' for s in D3B_RUN_ROOTS}
    for s,p in D3B_RUN_ROOTS.items(): shutil.copytree(p, IN_D3B[s])
    print('staged in', round(time.time()-t0), 's; disk free GB', round(shutil.disk_usage(ST).free/1e9,1))
args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',f'{OUT}/run','--family',FAMILY,'--d2-root',IN_D2,'--profile','production_official']+[x for s in sorted(IN_D3B) for x in ('--d3b-root', f'{s}={IN_D3B[s]}')]
rc=subprocess.run(args,capture_output=True,text=True); open(f'{OUT}/profile_stdout.txt','w').write(rc.stdout); open(f'{OUT}/profile_stderr.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
vp=f'{OUT}/run/d3c_profile_record_{FAMILY}.json'
try:
    v=json.load(open(vp))
    if not isinstance(v, dict) or not isinstance(v.get('D3C_PASS'), bool): raise ValueError('profile record is not a result dictionary with boolean D3C_PASS')
except Exception as read_error:
    v=dict(D3C_PASS=False, stage='RECORD_MISSING_OR_INVALID', gates={}, failures=[f'profile record unreadable: {read_error!r}', f'script returncode {rc.returncode}'], launcher_fallback=True, stderr_tail=rc.stderr[-2000:])
g=v.get('gate') or {}
print('rc', rc.returncode, '| stage', v.get('stage'), '| D3C_PASS', v.get('D3C_PASS'), '| gate passed', g.get('passed'), '| failures', v.get('failures'), '| rss MB', v.get('stages_rss_mb'), '| seconds', round(v.get('seconds', 0)))
profile_pass=bool(rc.returncode==0 and v.get('D3C_PASS') is True and g.get('passed') is True and not v.get('launcher_fallback'))
json.dump(dict(lock=lock, exit_code=rc.returncode, stage=v.get('stage'), D3C_PASS=v.get('D3C_PASS'), gate_passed=g.get('passed'), launcher_fallback=bool(v.get('launcher_fallback')), profile_pass=profile_pass, staged_inputs=dict(d2=IN_D2, d3b=IN_D3B), seconds_total=time.time()-t0), open(f'{OUT}/profile_final_record.json','w'), indent=1); print('profile_pass', profile_pass)


In [ ]:
# --- 3. zip the evidence (small: records, plan identity, lock, stdout/stderr) for the audit
zp=f'/content/d3c_profile_{FAMILY}_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=OUT)
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)
